In [4]:
pip install pandas openpyxl python-docx

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import re
import pandas as pd
import textract # For .doc and other formats
from docx import Document # For .docx specifically

# --- Configuration ---
DOC_FILES_DIRECTORY = "path/to/your/doc_files"  # Replace with the actual path
EXCEL_FILE_PATH = "path/to/your/spreadsheet.xlsx"  # Replace with your Excel file
OUTPUT_EXCEL_FILE = "path/to/your/output_adhesives_data.xlsx" # Desired output path

# --- REGEX DEFINITIONS ---
# This is the MOST CRITICAL part to customize.
# Keys are the EXACT column names from your Excel sheet.
# Values are regex patterns to find the corresponding value in the DOC text.
# The regex should have ONE capturing group for the value.
# Example: For a property "Viscosity", if it looks like "Viscosity: 100-200 cP"
# a regex could be r"Viscosity\s*:\s*(.*?)(?:\n|$)"
# This searches for "Viscosity:", then optional space, then captures everything (.*?)
# until a newline (\n) or end of text ($).

# YOU WILL LIKELY NEED TO ADJUST THESE REGEXES BASED ON YOUR DOCUMENT FORMATTING
PROPERTY_REGEX_MAP = {
    # Example Properties - REPLACE WITH YOUR ACTUAL EXCEL COLUMN NAMES AND REGEXES
    "Adhesive Name": r"(?:Product Name|Adhesive Name|Name)\s*[:\-=\s]\s*(.*?)(?:\n|$)",
    "Viscosity (cP)": r"Viscosity\s*[:\-=\s]\s*([\d\.\s-]+(?:cP|mPa\.s|Pa\.s))(?:\n|$)",
    "Cure Time (hours)": r"Cure Time\s*[:\-=\s]\s*([\d\.\s-]+(?:hours?|hrs?|minutes?|mins?))(?:\n|$)",
    "Tensile Strength (MPa)": r"Tensile Strength\s*[:\-=\s]\s*([\d\.\s-]+(?:MPa|psi|N/mm²))(?:\n|$)",
    "Operating Temperature (°C)": r"Operating Temperature(?: Range)?\s*[:\-=\s]\s*([\d\.\s\-\sto]+(?:°C|°F|C|F))(?:\n|$)",
    # Add more properties from your Excel sheet here
}

# Generic regex to find potential new properties (Key: Value type patterns)
# Captures: 1=Key, 2=Value
# This tries to find lines that look like "Some Property Name: Some Value"
# It avoids capturing if the key is too short (e.g. "A: B") or only numbers.
# It tries to stop before another similar pattern or known units for better value capture.
GENERIC_PROPERTY_REGEX = r"([A-Za-z][A-Za-z\s()\/]{3,})\s*[:\-]\s*(.*?)(?=\n[A-Za-z][A-Za-z\s()\/]{3,}\s*[:\-]|\n\n|$)"

# --- Helper Functions ---

def extract_text_from_file(filepath):
    """Extracts text from .doc or .docx files."""
    try:
        if filepath.endswith('.docx'):
            doc = Document(filepath)
            full_text = []
            for para in doc.paragraphs:
                full_text.append(para.text)
            # Consider extracting from tables too if needed (more complex)
            # for table in doc.tables:
            #     for row in table.rows:
            #         for cell in row.cells:
            #             full_text.append(cell.text)
            return '\n'.join(full_text)
        elif filepath.endswith('.doc'):
            # textract might use antiword or other tools.
            # Ensure they are installed if needed, or convert .doc to .docx manually.
            byte_text = textract.process(filepath)
            return byte_text.decode('utf-8', errors='ignore') # Or other relevant encoding
        else:
            print(f"Unsupported file type: {filepath}")
            return ""
    except Exception as e:
        print(f"Error extracting text from {filepath}: {e}")
        return ""

def clean_value(value):
    """Cleans up extracted value."""
    if value:
        return value.strip()
    return None

def find_property_value(text, regex_pattern):
    """Finds a property value using regex."""
    match = re.search(regex_pattern, text, re.IGNORECASE | re.DOTALL)
    if match:
        return clean_value(match.group(1)) # Assumes value is in the first capturing group
    return None

# --- Main Script ---
def main():
    # 1. Read Excel template to get column headers
    try:
        df_template = pd.read_excel(EXCEL_FILE_PATH)
        predefined_columns = list(df_template.columns)
        print(f"Predefined columns from Excel: {predefined_columns}")
    except FileNotFoundError:
        print(f"Error: Excel template file not found at {EXCEL_FILE_PATH}")
        print("Please ensure the Excel file exists and the path is correct.")
        print("If the Excel is just for column names, create an empty one with headers.")
        # Fallback: Use keys from PROPERTY_REGEX_MAP if Excel is not strictly for template reading
        # but more for guiding which properties to look for.
        # Or, if the Excel sheet is truly empty and just meant to BE populated,
        # then PROPERTY_REGEX_MAP keys are the primary source.
        # For this script, let's assume Excel provides the base columns.
        predefined_columns = list(PROPERTY_REGEX_MAP.keys())
        if not predefined_columns:
            print("Error: No predefined columns from Excel and PROPERTY_REGEX_MAP is empty. Exiting.")
            return
        print(f"Using columns from PROPERTY_REGEX_MAP as predefined: {predefined_columns}")


    # 2. List DOC files
    doc_files = []
    for root, _, files in os.walk(DOC_FILES_DIRECTORY):
        for file in files:
            if file.lower().endswith(('.doc', '.docx')):
                doc_files.append(os.path.join(root, file))

    if not doc_files:
        print(f"No .doc or .docx files found in {DOC_FILES_DIRECTORY}")
        return

    print(f"Found {len(doc_files)} documents to process.")

    all_adhesives_data = []
    all_discovered_properties = set(predefined_columns) # Keep track of all property names

    # 3. Iterate and Extract
    for i, doc_path in enumerate(doc_files):
        print(f"\nProcessing file {i+1}/{len(doc_files)}: {os.path.basename(doc_path)}...")
        text_content = extract_text_from_file(doc_path)

        if not text_content:
            print(f"Could not extract text from {os.path.basename(doc_path)}. Skipping.")
            # Add a row with filename and NULLs for other fields for completeness
            adhesive_data = {"Filename": os.path.basename(doc_path)}
            for col in predefined_columns:
                if col not in adhesive_data: # Avoid overwriting if already set
                     adhesive_data[col] = "NULL" # Or pd.NA
            all_adhesives_data.append(adhesive_data)
            continue

        adhesive_data = {"Filename": os.path.basename(doc_path)}

        # Extract predefined properties
        for prop_name, prop_regex in PROPERTY_REGEX_MAP.items():
            if prop_name in predefined_columns: # Only look for props that are expected columns
                value = find_property_value(text_content, prop_regex)
                adhesive_data[prop_name] = value if value else "NULL" # Or pd.NA

        # Attempt to find additional properties not in the predefined list
        # This is more experimental and may need tuning
        print("  Searching for generic properties...")
        discovered_in_this_doc = set()
        for match in re.finditer(GENERIC_PROPERTY_REGEX, text_content, re.IGNORECASE):
            potential_new_prop_name = clean_value(match.group(1))
            potential_value = clean_value(match.group(2))

            if potential_new_prop_name and potential_value and len(potential_new_prop_name) > 2:
                # Normalize property name (e.g., title case, remove extra spaces)
                normalized_prop_name = ' '.join(word.capitalize() for word in potential_new_prop_name.split())
                
                # Avoid re-processing if already found by specific regex or this generic one for this doc
                if normalized_prop_name not in adhesive_data and normalized_prop_name not in discovered_in_this_doc:
                    is_known_from_map = any(normalized_prop_name.lower() == k.lower() for k in PROPERTY_REGEX_MAP.keys())
                    
                    if not is_known_from_map: # Only add if truly "new" based on PROPERTY_REGEX_MAP
                        print(f"    Found potential new property: '{normalized_prop_name}' = '{potential_value[:50]}...'")
                        adhesive_data[normalized_prop_name] = potential_value
                        all_discovered_properties.add(normalized_prop_name)
                        discovered_in_this_doc.add(normalized_prop_name)

        # Ensure all predefined columns are present, even if not found in this doc
        for col in predefined_columns:
            if col not in adhesive_data:
                adhesive_data[col] = "NULL" # Or pd.NA
        
        all_adhesives_data.append(adhesive_data)

    # 4. & 5. Create DataFrame
    if not all_adhesives_data:
        print("No data was extracted from any documents.")
        return

    # Ensure all dictionaries in all_adhesives_data have all keys from all_discovered_properties
    # This makes DataFrame creation robust if some docs have props others don't
    final_data_for_df = []
    for record in all_adhesives_data:
        new_record = {}
        for prop_key in all_discovered_properties: # Use the master list of all props
            new_record[prop_key] = record.get(prop_key, "NULL") # Or pd.NA
        final_data_for_df.append(new_record)
        
    # Order columns: Filename first, then predefined columns, then new columns alphabetically
    ordered_columns = ["Filename"]
    # Add predefined columns in their original order, if they were actually found
    for col in predefined_columns:
        if col in all_discovered_properties and col != "Filename":
            ordered_columns.append(col)
    # Add new discovered columns alphabetically
    newly_added_cols = sorted(list(all_discovered_properties - set(ordered_columns)))
    ordered_columns.extend(newly_added_cols)
    
    df_output = pd.DataFrame(final_data_for_df, columns=ordered_columns)


    # 6. Handle Missing Data (already done with "NULL" or pd.NA)
    # If you used pd.NA and want "NULL" string:
    # df_output = df_output.fillna("NULL")

    # 7. Export to Excel
    try:
        df_output.to_excel(OUTPUT_EXCEL_FILE, index=False)
        print(f"\nSuccessfully extracted data and saved to {OUTPUT_EXCEL_FILE}")
    except Exception as e:
        print(f"\nError saving data to Excel: {e}")
        print("You might want to save to a CSV as a backup:")
        try:
            csv_output_file = OUTPUT_EXCEL_FILE.replace('.xlsx', '.csv')
            df_output.to_csv(csv_output_file, index=False)
            print(f"Backup saved to {csv_output_file}")
        except Exception as e_csv:
            print(f"Could not save CSV backup: {e_csv}")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'textract'